## Agentic Framework: Ramp-Up Training Materials

To get the latest version of AF Python package, use:

``` bash
pip install --upgrade agent-framework --pre
pip install --upgrade azure-ai-projects --pre
```

## 📒 Notebook 0: Quick Start

### 🪜 Step 1: Configure environment

In [1]:
# Suppress warnings from agent_framework_azure module
import warnings
warnings.filterwarnings(
    action = "ignore",
    category = UserWarning,
    module = "agent_framework_azure"
)

In [2]:
# Import required packages
import os
import asyncio
from agent_framework import ChatAgent
from agent_framework.azure import AzureAIClient
from azure.identity.aio import DefaultAzureCredential

In [3]:
# Fix: Disable Accept-Encoding to prevent compressed responses
import azure.core.pipeline.policies as policies

# Store original
_original_on_request = policies.HeadersPolicy.on_request

def patched_on_request(self, request):
    """Remove Accept-Encoding header to prevent compressed responses."""
    _original_on_request(self, request)
    # Tell server we don't accept compressed responses
    request.http_request.headers['Accept-Encoding'] = 'identity'

policies.HeadersPolicy.on_request = patched_on_request
print("Applied Accept-Encoding fix to disable compression")

Applied Accept-Encoding fix to disable compression


In [4]:
# Set environment variables
PROJECT_ENDPOINT = os.environ.get("AZURE_FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT = os.environ.get("AZURE_FOUNDRY_GPT_MODEL")

if not PROJECT_ENDPOINT or not MODEL_DEPLOYMENT:
    print("WARNING: Environment variables not set properly!")
else:
    print("Environment variables set successfully!")

Environment variables set successfully!


### 🪜 Step 2: Create AI agent

In [5]:
# Define AI client
ai_client = AzureAIClient(
    agent_name = "MAFSDK-Agentv2-Haiku",
    project_endpoint = PROJECT_ENDPOINT,
    model_deployment_name = MODEL_DEPLOYMENT,
    async_credential = DefaultAzureCredential()
)

print(f"Created AI client: {ai_client.agent_name}")

Created AI client: MAFSDK-Agentv2-Haiku


In [6]:
# Define chat agent
agent = ChatAgent(
    chat_client = ai_client,
    name = "haiku-poet-agent",
    instructions = "You are a haiku poet. Respond to user queries with a haiku."
)

print(f"Created chat agent: {agent.name}")

Created chat agent: haiku-poet-agent


In [7]:
# Define alternative agent's constructor
agent_alt = ai_client.create_agent(
    name = "alternative-haiku-poet-agent",
    instructions = "You are a haiku poet. Respond to user queries with a haiku."
)

print(f"Created alternative chat agent: {agent_alt.name}")

Created alternative chat agent: alternative-haiku-poet-agent


### 🪜 Step 3: Run the agent

In [8]:
# Test chat agent with a prompt
prompt = "What is life?"
print(f"User: {prompt}\n")

response = await agent.run(prompt)
print(f"Agent: {response.text}\n")

User: What is life?

Agent: River flows onward,  
Moments bloom, then drift away—  
Life, a fleeting dance.



In [9]:
# Test streaming of alternative agent with a prompt
prompt_alt = "Describe the ocean."
print(f"User: {prompt_alt}\n")

print("Agent:")
async for streaming_update in agent_alt.run_stream(prompt_alt):
    if streaming_update.text:
        print(streaming_update.text, end="", flush=True)

User: Describe the ocean.

Agent:
Endless waves whisper,  
Deep blue heart in sun’s embrace,  
Secrets in salt breath.

### 🪜 Step 4: Housekeeping

In [ ]:
# Delete Azure AI client
await ai_client.close()

print("AI client closed successfully.")